# Llama 3.2 Multimodal  for Text and image data setimental analysis

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [10]:
# !pip install wolframalpha==5.1.3
# !pip install together==1.2.0
# !pip install llama-stack==0.0.36 
# !pip install llama-stack-client==0.0.35 
!pip install gradio==4.43.0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
conda-repo-cli 1.0.4 requires pathlib, which is not installed.
anaconda-project 0.10.2 requires ruamel-yaml, which is not installed.
opensearch-py 2.2.0 requires urllib3<2,>=1.21.1, but you have urllib3 2.2.3 which is incompatible.
langchain 0.0.162 requires pydantic<2,>=1, but you have pydantic 2.9.2 which is incompatible.
elastic-transport 8.4.0 requires urllib3<2,>=1.26.2, but you have urllib3 2.2.3 which is incompatible.
botocore 1.24.32 requires urllib3<1.27,>=1.25.4, but you have urllib3 2.2.3 which is incompatible.

  Attempting uninstall: urllib3
    Found existing installation: urllib3 1.26.9
    Uninstalling urllib3-1.26.9:
      Successfully uninstalled urllib3-1.26.9
  Attempting uninstall: httpcore
    Found existing installation: httpcore 0.16.3
    Uninstalling httpcore-0.16.3:
      Successfully uninstalled httpcore-0.16.3
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2022.2.0
    Uninstalling fsspec-2022.2.0:
      Successfully uninstalled fsspec-2022.2.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.14.1
    Uninstalling huggingface-hub-0.14.1:
      Successfully uninstalled huggingface-hub-0.14.1
  Attempting uninstall: httpx
    Found existing installation: httpx 0.23.3
    Uninstalling httpx-0.23.3:
      Successfully uninstalled httpx-0.23.3



argilla 1.6.0 requires httpx<0.24,>=0.15, but you have httpx 0.27.2 which is incompatible.
argilla 1.6.0 requires numpy<1.24.0, but you have numpy 1.24.4 which is incompatible.


# requirements 
#all the pip install needs

#python 3.10.14
numpy==1.26.4
together==1.2.0
python-dotenv~=1.0.1
tiktoken==0.7.0
blobfile==3.0.0
torch==2.4.0
matplotlib==3.9.2
wolframalpha==5.1.3
tavily-python==0.5.0
llama-stack==0.0.36
llama-stack-client==0.0.35
gradio==4.43.0


In [3]:
from utils import load_env
load_env()

## Text input only question

In [50]:
# https://api.together.xyz/ --->  sign in to generate your api key using gmail id.

In [49]:
api_key= open("api_key.txt", "r")
api_key=api_key.read()
# print(api_key)

In [42]:
import requests
import json

from dotenv import load_dotenv, find_dotenv
import os
from wolframalpha import Client

def llama32(messages, model_size=11):
  model = f"meta-llama/Llama-3.2-{model_size}B-Vision-Instruct-Turbo"
  url = f"{os.getenv('DLAI_TOGETHER_API_BASE', 'https://api.together.xyz')}/v1/chat/completions"
  payload = {
    "model": model,
    "max_tokens": 4096,
    "temperature": 0.0,
    "stop": ["<|eot_id|>","<|eom_id|>"],
    "messages": messages
  }

  headers = {
    "Accept": "application/json",
    "Content-Type": "application/json",
    "Authorization": f"Bearer {api_key}"
  }
  res = json.loads(requests.request("POST", url, headers=headers, data=json.dumps(payload)).content)

  if 'error' in res:
    raise Exception(res['error'])

  return res['choices'][0]['message']['content']

In [47]:
messages = [
  {"role": "user",
    "content":  [
      {"type": "text",
        "text": "find the sentiment of the given sentence"
      },
      {"type": "text",
        "text": "Elon mask robotaxi and robovan is stunning the entire industy"
      }
    ]
  }
]

In [48]:
response_32 = llama32(messages, 90)
print(response_32)

A very positive sentence!

The sentiment of the given sentence is: **EXTREMELY POSITIVE**

Here's a breakdown of the sentiment analysis:

* The word "stunning" has a very strong positive connotation, indicating a high level of admiration and amazement.
* The phrase "the entire industry" suggests that the impact of Elon Musk's robotaxi and robovan is widespread and significant.
* The tone of the sentence is one of excitement and awe, with no negative words or phrases to balance out the sentiment.

Overall, the sentiment of this sentence is one of unbridled enthusiasm and admiration for Elon Musk's innovative creations.


## Question about an image

### Using a local image

In [33]:
import base64

def encode_image(image_path):
  with open(image_path, "rb") as image_file:
    return base64.b64encode(image_file.read()).decode('utf-8')
base64_image = encode_image(r"C:\Users\HP\Downloads\elonmask_robotaxi.png")

In [34]:
base64_image

'...'

In [37]:
messages = [
  {"role": "user",
    "content": [
      {"type": "text",
        "text": "describe the sentiment of the images text"
      },
      {"type": "image_url",
        "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}
      }
    ]
  },
]

In [38]:
# disp_image(image_url)
result = llama32(messages,90)
print(result)

The sentiment of the images' text is positive, as it expresses admiration for the interior design and smartness of Tesla. The user also mentions that Elon Musk is already in the future, implying a sense of innovation and progress. Overall, the tone is enthusiastic and optimistic, suggesting that the user is excited about the potential of Tesla and its technology.
